# ShikshaSetu Foundation — Challenge A Exploratory Data Analysis
**Veltris Vertical AI Academy Industry Challenge Assessment**

This notebook reproduces every calculation on the exact assessment datasets:
- **Dataset A1**: 10 schools, 900 enrolled students, average attendance.
- **Dataset A2**: 12 synthetic illustrative student records (`ST001`–`ST012`).
- **Dataset A3**: 5 outreach workers, max 4 visits/worker/day, 5 working days/week, 100 max weekly visits, INR 25,000 weekly budget (INR 250/visit).

> **Integrity Boundary**: Dataset A2 contains synthetic records with **no verified dropout outcome labels**. This notebook implements transparent rule-based prioritization and descriptive correlation checks—never claiming a trained dropout classifier or causal proof.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Exact Dataset A1
schools_df = pd.DataFrame([
    {"School": "School 1", "Enrolled": 100, "Attendance_Pct": 82},
    {"School": "School 2", "Enrolled": 95, "Attendance_Pct": 76},
    {"School": "School 3", "Enrolled": 90, "Attendance_Pct": 72},
    {"School": "School 4", "Enrolled": 85, "Attendance_Pct": 68},
    {"School": "School 5", "Enrolled": 100, "Attendance_Pct": 65},
    {"School": "School 6", "Enrolled": 80, "Attendance_Pct": 80},
    {"School": "School 7", "Enrolled": 90, "Attendance_Pct": 61},
    {"School": "School 8", "Enrolled": 85, "Attendance_Pct": 74},
    {"School": "School 9", "Enrolled": 90, "Attendance_Pct": 70},
    {"School": "School 10", "Enrolled": 85, "Attendance_Pct": 62},
])

total_enrolled = schools_df["Enrolled"].sum()
weighted_att = (schools_df["Enrolled"] * schools_df["Attendance_Pct"]).sum() / total_enrolled
print(f"Total Schools: {len(schools_df)} | Total Enrolled: {total_enrolled} | Weighted Avg Attendance: {weighted_att:.2f}%")
schools_df.sort_values("Attendance_Pct")

In [ ]:
# Exact Dataset A2 (12 synthetic student records)
students_df = pd.DataFrame([
    {"ID": "ST001", "School": 1, "Attendance": 92, "Prev_Score": 78, "Distance_km": 1, "Prior_Visits": 0},
    {"ID": "ST002", "School": 1, "Attendance": 54, "Prev_Score": 62, "Distance_km": 4, "Prior_Visits": 1},
    {"ID": "ST003", "School": 2, "Attendance": 68, "Prev_Score": 71, "Distance_km": 3, "Prior_Visits": 0},
    {"ID": "ST004", "School": 3, "Attendance": 41, "Prev_Score": 45, "Distance_km": 6, "Prior_Visits": 2},
    {"ID": "ST005", "School": 3, "Attendance": 83, "Prev_Score": 81, "Distance_km": 2, "Prior_Visits": 0},
    {"ID": "ST006", "School": 4, "Attendance": 59, "Prev_Score": 52, "Distance_km": 5, "Prior_Visits": 1},
    {"ID": "ST007", "School": 5, "Attendance": 48, "Prev_Score": 58, "Distance_km": 7, "Prior_Visits": 0},
    {"ID": "ST008", "School": 6, "Attendance": 88, "Prev_Score": 85, "Distance_km": 1, "Prior_Visits": 0},
    {"ID": "ST009", "School": 7, "Attendance": 35, "Prev_Score": 39, "Distance_km": 8, "Prior_Visits": 2},
    {"ID": "ST010", "School": 8, "Attendance": 72, "Prev_Score": 68, "Distance_km": 3, "Prior_Visits": 0},
    {"ID": "ST011", "School": 9, "Attendance": 57, "Prev_Score": 60, "Distance_km": 4, "Prior_Visits": 1},
    {"ID": "ST012", "School": 10, "Attendance": 44, "Prev_Score": 48, "Distance_km": 6, "Prior_Visits": 0},
])

print("Descriptive Correlations (NOT Causation):")
print(students_df[["Attendance", "Prev_Score", "Distance_km", "Prior_Visits"]].corr().round(3))

In [ ]:
# Transparent Prototype Prioritization Scoring (matching backend app/intelligence/risk_engine.py)
def compute_priority(row):
    att_pts = ((100 - row["Attendance"]) / 100.0) * 40.0
    acad_pts = ((100 - row["Prev_Score"]) / 100.0) * 30.0
    dist_pts = min(row["Distance_km"] / 10.0, 1.0) * 20.0
    # Prior visits only trigger a follow-up check if outcomes are unrecorded
    followup_pts = 10.0 if row["Prior_Visits"] > 0 else 0.0
    return round(att_pts + acad_pts + dist_pts + followup_pts, 2)

students_df["Priority_Score"] = students_df.apply(compute_priority, axis=1)
students_df["Band"] = np.where(
    (students_df["Priority_Score"] >= 60) | (students_df["Attendance"] < 50),
    "High Priority",
    np.where((students_df["Priority_Score"] >= 40) | (students_df["Attendance"] < 65), "Review Soon", "Routine Review")
)
students_df.sort_values("Priority_Score", ascending=False)

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ordered_schools = schools_df.sort_values("Attendance_Pct")
colors = ["#D97998" if x < 70 else "#4D9B78" for x in ordered_schools["Attendance_Pct"]]
axes[0].barh(ordered_schools["School"], ordered_schools["Attendance_Pct"], color=colors)
axes[0].set_title("Dataset A1: School Average Attendance (%)")
axes[0].set_xlabel("Attendance %")

axes[1].scatter(students_df["Distance_km"], students_df["Attendance"], c="#D97998", s=90)
for _, r in students_df.iterrows():
    axes[1].annotate(r["ID"], (r["Distance_km"] + 0.12, r["Attendance"] + 0.8), fontsize=8)
axes[1].set_title("Dataset A2: Distance vs. Attendance (Correlation != Causation)")
axes[1].set_xlabel("Distance to School (km)")
axes[1].set_ylabel("Attendance (%)")
plt.tight_layout()
plt.show()